# Etap 1 - Pozyskanie danych z cBioPortal API

**Projekt:** Analiza przeżywalności pacjentów z glejakami w kohorcie TCGA  
**Autor:** Anna Zimniewicz  
**Data:** 18 maja 2026

## Cel notebooka

Pobranie surowych danych klinicznych i molekularnych dla dwóch zbiorów TCGA z cBioPortal:
- **TCGA-GBM** (Glioblastoma Multiforme, Firehose Legacy) - 619 próbek
- **TCGA-LGG** (Brain Lower Grade Glioma, Firehose Legacy) - 530 próbek

Dane zostaną zapisane jako surowe pliki CSV w folderze `data/raw/`.

## Plan

1. Konfiguracja: import bibliotek, ustawienie bazowego URL API
2. Sprawdzenie statusu serwera cBioPortal (`/api/health`)
3. Pobranie listy wszystkich studies i znalezienie `studyId` dla naszych dwóch kohort
4. Pobranie danych klinicznych dla obu studies
5. Pobranie mutacji dla genów IDH1, IDH2
6. Zapis surowych danych do `../data/raw/`

## Źródło danych

- cBioPortal: https://www.cbioportal.org
- Dokumentacja API: https://www.cbioportal.org/api/swagger-ui/index.html

In [1]:
# Import bibliotek potrzebnych do pobierania i obróbki danych
import requests              # wysyłanie zapytań HTTP do API
import pandas as pd          # praca z danymi tabelarycznymi 
from pathlib import Path     # nowoczesna obsługa ścieżek plików 
from datetime import datetime  # do oznaczania daty pobrania danych

# Sprawdzamy, czy biblioteki się załadowały
print("Wszystkie biblioteki zaimportowane poprawnie.")
print(f"pandas version: {pd.__version__}")
print(f"requests version: {requests.__version__}")

Wszystkie biblioteki zaimportowane poprawnie.
pandas version: 3.0.2
requests version: 2.33.1


In [2]:
# === KONFIGURACJA ===

# Bazowy URL cBioPortal API
# Wszystkie endpointy budujemy doklejając kolejne fragmenty do tego URL-a
BASE_URL = "https://www.cbioportal.org/api"

# Identyfikatory dwóch studies, które nas interesują
# (te ID znajdziemy za chwilę przez API - na razie wpisujemy je z głowy na podstawie znajomości cBioPortal)
STUDY_IDS = {
    "GBM": "gbm_tcga",       # Glioblastoma Multiforme (TCGA, Firehose Legacy)
    "LGG": "lgg_tcga",       # Brain Lower Grade Glioma (TCGA, Firehose Legacy)
}

# Ścieżka do folderu na surowe dane
# Path("..") oznacza "folder wyżej" - bo notebook jest w notebooks/, a data jest w głównym folderze projektu
RAW_DATA_DIR = Path("..") / "data" / "raw"

# Upewniamy się, że folder istnieje (jeśli nie - utwórz)
# parents=True: utwórz też foldery nadrzędne, jeśli nie istnieją
# exist_ok=True: nie rzucaj błędu, jeśli folder już jest
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Data pobrania - przyda się do nazw plików i dokumentacji
DOWNLOAD_DATE = datetime.now().strftime("%Y-%m-%d")

# Wypisujemy konfigurację dla kontroli
print(f"Base URL: {BASE_URL}")
print(f"Studies: {STUDY_IDS}")
print(f"Folder docelowy: {RAW_DATA_DIR.resolve()}")  # .resolve() pokazuje pełną ścieżkę
print(f"Data pobrania: {DOWNLOAD_DATE}")

Base URL: https://www.cbioportal.org/api
Studies: {'GBM': 'gbm_tcga', 'LGG': 'lgg_tcga'}
Folder docelowy: C:\Users\anazi\Desktop\praca_dyplomowa\data\raw
Data pobrania: 2026-05-19


In [3]:
# === KROK 1: Sprawdzenie statusu serwera cBioPortal ===

# Budujemy URL endpointu przez sklejenie bazowego URL-a z konkretną ścieżką
health_url = f"{BASE_URL}/health"

# Wysyłamy zapytanie GET
# requests.get() zwraca obiekt Response zawierający odpowiedź serwera
response = requests.get(health_url)

# Sprawdzamy, czy zapytanie się powiodło
# raise_for_status() rzuca wyjątek, jeśli status_code nie jest w zakresie 200-299
response.raise_for_status()

# Wyświetlamy informacje diagnostyczne
print(f"URL zapytania: {health_url}")
print(f"Status code: {response.status_code}")
print(f"Czas odpowiedzi: {response.elapsed.total_seconds()} s")
print(f"Treść odpowiedzi: {response.text}")

URL zapytania: https://www.cbioportal.org/api/health
Status code: 200
Czas odpowiedzi: 1.368747 s
Treść odpowiedzi: {"status":"UP"}


In [5]:
# === KROK 2: Pobranie listy wszystkich studies w cBioPortal ===

# Endpoint /studies zwraca listę wszystkich dostępnych badań w cBioPortal
# Dodajemy parametr projection=SUMMARY - dostaniemy skrócone metadane (bez wszystkich detali)

studies_url = f"{BASE_URL}/studies" 
params = {
    "projection" : "SUMMARY", 
    "pageSize" : 10_000_000,
    "pageNumber" : 0,
    "direction" : "ASC",
}

response = requests.get(studies_url, params=params)
response.raise_for_status()

studies_data = response.json()

print(f"Liczba studies w cBioPortal: {len(studies_data)}")
print(f"Typ obiektu: {type(studies_data)}")
print(f"Typ pierwszego elementu: {type(studies_data[0])}")
print(f"\nKlucze w pierwszym elemencie:")
print(list(studies_data[0].keys()))



Liczba studies w cBioPortal: 535
Typ obiektu: <class 'list'>
Typ pierwszego elementu: <class 'dict'>

Klucze w pierwszym elemencie:
['name', 'description', 'publicStudy', 'groups', 'status', 'importDate', 'allSampleCount', 'readPermission', 'resourceCounts', 'studyId', 'cancerTypeId', 'referenceGenome']


In [13]:
# === KROK 3: Konwersja na DataFrame i filtrowanie do naszych studies ===
# pandas potrafi zamienić listę słowników bezpośrednio w DataFrame
# Każdy słownik → jeden wiersz, klucze słownika → nazwy kolumn

studies_df = pd.DataFrame(studies_data)

print(f"Wymiary tabeli: {studies_df.shape}") #.shape zwraca (liczba_wierszy, liczba_kolumn)
print(f"Kolumny: {list(studies_df.columns)}")

print("\nTrzy pierwsze studies (wybrane kolumny):")
display(studies_df[["studyId", "name", "cancerTypeId", "allSampleCount"]].head(3)) #.head(3) zwraca 3 pierwsze wiersze 


Wymiary tabeli: (535, 14)
Kolumny: ['name', 'description', 'publicStudy', 'groups', 'status', 'importDate', 'allSampleCount', 'readPermission', 'resourceCounts', 'studyId', 'cancerTypeId', 'referenceGenome', 'pmid', 'citation']

Trzy pierwsze studies (wybrane kolumny):


,studyId,name,cancerTypeId,allSampleCount
0,breast_cptac_gdc,"Breast Cancer (CPTAC GDC, 2025)",breast,1
1,glioma_mskcc_2019,"Glioma (MSK, Clin Cancer Res 2019)",difg,1
2,os_target_gdc,"Osteosarcoma (TARGET GDC, 2025)",os,1


Wyjaśnienie: te trzy wiersze to nowo dodane studies w stanie roboczym ("placeholdery" z 1 testową próbką). Sortowanie alfabetyczne wrzuciło je na początek. Nasze GBM (619 próbek) i LGG (530 próbek) są gdzieś dalej w tabeli.

In [16]:
# === KROK 4: Filtrowanie do naszych dwóch studies (GBM + LGG Firehose Legacy) ===

# Lista ID, których szukamy (bierzemy je ze słownika STUDY_IDS)
target_ids = list(STUDY_IDS.values())  # ['gbm_tcga', 'lgg_tcga'] #.values zwraca wartości słownika, list zamienia do na listę
print(f"Szukamy studies o ID: {target_ids}")

# Filtrowanie DataFrame metodą .isin()
# Tworzymy warunek logiczny: które wiersze mają studyId w naszej liście?
#Weź kolumnę studyId z DataFrame. 
#Dla każdego wiersza sprawdź, czy wartość w tej kolumnie znajduje się w liście target_ids. 
#Zwróć kolumnę z True/False.
mask = studies_df["studyId"].isin(target_ids)

our_studies = studies_df[mask]

print(f"\nZnaleziono {len(our_studies)} pasujących studies:")
display(our_studies[["studyId", "name", "cancerTypeId", "allSampleCount", "referenceGenome"]])

Szukamy studies o ID: ['gbm_tcga', 'lgg_tcga']

Znaleziono 2 pasujących studies:


,studyId,name,cancerTypeId,allSampleCount,referenceGenome
95,gbm_tcga,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",difg,1,hg19
227,lgg_tcga,"Brain Lower Grade Glioma (TCGA, Firehose Legacy)",difg,1,hg19


allSampleCount się nie zgadza 

In [17]:
# === DIAGNOSTYKA: Sprawdzenie metadanych dla naszych studies z DETAILED projection ===

# Powtarzamy zapytanie, ale tym razem z projection=DETAILED
# Powinniśmy dostać pełne metadane, w tym prawdziwy allSampleCount
params_detailed = {
    "projection": "DETAILED",
    "pageSize": 10_000_000,
    "pageNumber": 0,
    "direction": "ASC",
}

response = requests.get(f"{BASE_URL}/studies", params=params_detailed)
response.raise_for_status()
studies_detailed = pd.DataFrame(response.json())

print(f"Liczba kolumn z DETAILED: {len(studies_detailed.columns)}")
print(f"Kolumny: {list(studies_detailed.columns)}")

# Filtrujemy do naszych dwóch studies
our_studies_detailed = studies_detailed[studies_detailed["studyId"].isin(target_ids)]

print("\nNasze studies z DETAILED projection:")
display(our_studies_detailed)

Liczba kolumn z DETAILED: 27
Kolumny: ['name', 'description', 'publicStudy', 'groups', 'status', 'importDate', 'allSampleCount', 'sequencedSampleCount', 'cnaSampleCount', 'mrnaRnaSeqSampleCount', 'mrnaRnaSeqV2SampleCount', 'mrnaMicroarraySampleCount', 'miRnaSampleCount', 'methylationHm27SampleCount', 'rppaSampleCount', 'massSpectrometrySampleCount', 'completeSampleCount', 'readPermission', 'treatmentCount', 'structuralVariantCount', 'resourceCounts', 'studyId', 'cancerTypeId', 'cancerType', 'referenceGenome', 'pmid', 'citation']

Nasze studies z DETAILED projection:


,name,description,publicStudy,groups,status,importDate,allSampleCount,sequencedSampleCount,cnaSampleCount,mrnaRnaSeqSampleCount,...,readPermission,treatmentCount,structuralVariantCount,resourceCounts,studyId,cancerTypeId,cancerType,referenceGenome,pmid,citation
95,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",TCGA Glioblastoma Multiforme. Source data from...,True,PUBLIC,0,2026-01-07 12:59:53,1,290,577,0,...,True,0,0,[],gbm_tcga,difg,"{'name': 'Diffuse Glioma', 'dedicatedColor': '...",hg19,NaN,NaN
227,"Brain Lower Grade Glioma (TCGA, Firehose Legacy)",TCGA Brain Lower Grade Glioma. Source data fro...,True,PUBLIC,0,2026-01-07 15:31:49,1,286,513,0,...,True,0,0,[],lgg_tcga,difg,"{'name': 'Diffuse Glioma', 'dedicatedColor': '...",hg19,NaN,NaN


## ⚠️ Uwaga diagnostyczna - bug w cBioPortal API

Endpoint `/studies` zwraca `allSampleCount: 1` dla obu naszych studies (`gbm_tcga`, `lgg_tcga`), 
zarówno z `projection=SUMMARY` jak i `DETAILED`. To jest **bug w metadanych cBioPortal** - na stronie 
webowej widać poprawne liczby (619 dla GBM, 530 dla LGG), ale licznik w API się rozjechał.

**Inne pola w DETAILED zawierają prawdziwe liczby per typ danych:**
- `sequencedSampleCount`: 290 (GBM), 286 (LGG) - próbki z danymi mutacji
- `cnaSampleCount`: 577 (GBM), 513 (LGG) - próbki z danymi CNA

**Realna wielkość kohorty zostanie ustalona** na podstawie liczby unikalnych pacjentów 
w pobranych danych klinicznych (krok 5).

In [23]:
# === KROK 5: Pobranie listy pacjentów dla obu studies ===

# Endpoint /studies/{studyId}/patients zwraca listę wszystkich pacjentów w danym studium
# Robimy to w pętli dla obu naszych studies (GBM i LGG)

patients_per_study = {}  # tu będziemy zbierać wyniki

for label, study_id in STUDY_IDS.items():
    # label = "GBM" lub "LGG", study_id = "gbm_tcga" lub "lgg_tcga"
    
    # Budujemy URL z konkretnym study_id
    url = f"{BASE_URL}/studies/{study_id}/patients"
    
    # Wysyłamy zapytanie z dużym pageSize, żeby dostać wszystko naraz
    params = {"pageSize": 100_000, "pageNumber": 0}
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    # Parsujemy odpowiedź i zamieniamy na DataFrame
    patients = pd.DataFrame(response.json())
    
    # Zapisujemy do słownika pod kluczem "GBM" / "LGG"
    patients_per_study[label] = patients
    
    print(f"{label} ({study_id}): {len(patients)} pacjentów")
    
print(f"\nŁącznie pacjentów: {sum(len(df) for df in patients_per_study.values())}")
print("\nKolumny w DataFrame pacjentów GBM:")
print(list(patients_per_study["GBM"].columns))
print("\nPierwsze 3 wiersze GBM:")
display(patients_per_study["GBM"].head(3))

GBM (gbm_tcga): 606 pacjentów
LGG (lgg_tcga): 516 pacjentów

Łącznie pacjentów: 1122

Kolumny w DataFrame pacjentów GBM:
['uniquePatientKey', 'patientId', 'studyId']

Pierwsze 3 wiersze GBM:


,uniquePatientKey,patientId,studyId
0,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga
1,VENHQS0wMi0wMDAzOmdibV90Y2dh,TCGA-02-0003,gbm_tcga
2,VENHQS0wMi0wMDA2OmdibV90Y2dh,TCGA-02-0006,gbm_tcga


---

## Stan na koniec sesji 1

**Zweryfikowano:**
- Serwer cBioPortal odpowiada poprawnie (`/health` → 200)
- Studies `gbm_tcga` i `lgg_tcga` istnieją i są to TCGA Firehose Legacy
- Realne kohorty: **GBM = 606 pacjentów, LGG = 516 pacjentów** (razem 1122)

**Do zrobienia w następnej sesji:**
- Pobranie danych klinicznych (`/clinical-data`) - 60+ zmiennych per pacjent
- Pobranie mutacji genów IDH1, IDH2 - dla weryfikacji statusu IDH
- Pobranie statusu MGMT methylation
- Zapis wszystkich surowych danych do `data/raw/`